# IMU Road Quality — worked example

Runs the full pipeline (load -> orientation-invariant vertical acceleration -> windowing -> fixed-scale roughness scoring/classification -> map) on **`Fotouni_Douala_26_03_2026/2026-03-26_08-44-46`**, the longest recorded drive and the only one confirmed by manual inspection to cover all four road classes (paved, unpaved, bad, very_bad).

See `reports/report.md` for the full methodology and `README.md` for the fixed `[0, 1]` roughness-score scale used here (0.25 / 0.5 / 0.75 are the same class boundaries on every drive, not re-fit per recording).

Run this notebook with the project's own environment: `uv run --with jupyter jupyter lab notebooks/fotouni_douala_example.ipynb` from the `IMURoadQuality/` directory.

In [ ]:
from pathlib import Path

import pandas as pd

from imu_road_quality.classification import ClassificationConfig, RoadClass, classify_drive, compute_score_bands
from imu_road_quality.io import load_drive
from imu_road_quality.mapping import build_map
from imu_road_quality.segments import build_segments
from imu_road_quality.signal_processing import compute_vertical_acceleration, window_features

REPO_ROOT = Path.cwd().parent  # IMURoadQuality/
DRIVE_PATH = REPO_ROOT / "data" / "car_cam" / "Fotouni_Douala_26_03_2026" / "2026-03-26_08-44-46"
DRIVE_NAME = "Fotouni_Douala_26_03_2026/2026-03-26_08-44-46"

DRIVE_PATH

## 1. Load the raw drive

Only `Accelerometer.csv`, `Gravity.csv`, and `Location.csv` are loaded by default (the pipeline doesn't use the heavier Gyroscope/TotalAcceleration/Orientation exports).

In [ ]:
drive = load_drive(DRIVE_PATH, name=DRIVE_NAME)
print(f"accelerometer rows: {len(drive.accelerometer):,}")
print(f"gravity rows:       {len(drive.gravity):,}")
print(f"location fixes:     {len(drive.location):,}")
drive.accelerometer.head()

## 2. Orientation-invariant vertical acceleration

Projects the raw accelerometer vector onto the Android Gravity sensor's instantaneous gravity direction, `a_vertical = (a . g) / |g|`, then removes the ~9.81 m/s² gravity offset so the signal oscillates around zero (see `reports/report.md` §2).

In [ ]:
vert_accel = compute_vertical_acceleration(drive)
vert_accel.plot(x="seconds_elapsed", y="vert_accel_dynamic", figsize=(12, 3), lw=0.4, legend=False, title="Dynamic vertical acceleration (m/s²)")

## 3. Window onto GPS fixes

Aggregates the ~100 Hz vertical-acceleration stream into one window per GPS fix (~1-2 Hz), computing std/RMS/peak-to-peak as vibration-energy features.

In [ ]:
windowed = window_features(vert_accel, drive.location)
scoring_input = windowed[windowed["speed"] >= 1.0].dropna(subset=["accel_std", "speed"]).copy()
print(f"windows: {len(windowed):,}, usable (speed >= 1 m/s): {len(scoring_input):,}")
scoring_input[["seconds_elapsed", "speed", "accel_std", "accel_rms", "accel_ptp"]].describe()

## 4. Fixed-scale roughness score and classification

`ClassificationConfig()` defaults are the same for every drive: speed-compensate `accel_std` (divide by `speed^1.5`), log-compress, and min-max normalize against **fixed** calibration constants (`score_min=0.02`, `score_max=0.58`, pooled once across the project's dataset — not fit on this drive). Classes are then cut at the fixed thresholds 0.25 / 0.5 / 0.75, identical on every drive.

In [ ]:
config = ClassificationConfig()  # fixed defaults, no per-drive tuning
classified, thresholds = classify_drive(scoring_input, config)
print(f"fixed thresholds: {thresholds}")

counts = classified["road_class"].value_counts()
for rc in RoadClass:
    n = int(counts.get(rc, 0))
    print(f"  {rc.value:>9s}: {n:5d}  ({n / len(classified):.0%})")

This drive is the one confirmed by manual inspection to cover **all four** road classes end to end, which is why it's used as the worked example here rather than one of the other four retained drives.

In [ ]:
classified.plot(
    x="seconds_elapsed", y="roughness_score", figsize=(12, 3), lw=0.6, legend=False,
    title="Roughness score over the drive (fixed 0.25/0.5/0.75 class boundaries)",
)
for thr in thresholds:
    import matplotlib.pyplot as plt
    plt.axhline(thr, color="black", lw=0.5, ls=":")

## 5. Build segments and the interactive map

In [ ]:
segments = build_segments(classified)
score_bands = compute_score_bands(classified)
print(f"segments: {len(segments)}")
score_bands

In [ ]:
fmap = build_map(segments, score_bands, drive_name=DRIVE_NAME)
fmap

## 6. Save outputs

Same outputs as `experiments/Cameroon/run_all.py` produces for every drive.

In [ ]:
out_dir = REPO_ROOT / "experiments" / "Cameroon" / "outputs" / DRIVE_NAME.replace("/", "__")
out_dir.mkdir(parents=True, exist_ok=True)
fmap.save(str(out_dir / "road_quality_map.html"))
classified.to_csv(out_dir / "classified_points.csv", index=False)
print(f"saved to {out_dir}")